In [6]:
from collections.abc import Iterator

import dnnlpy
import torch
import torch.utils.data as utils
from jinja2 import optimizer
from torch import Tensor

print('PyTorch version:', torch.__version__)

device = dnnlpy.get_default_device()
print('PyTorch device:', device)

"""
    - Dataset: 用统一接口访问样本
    - DataLoader: 把样本组织成mini-batch
    - collate_fn: 样本如何拼成batch
    - num_workers: 让数据加载和模型计算并行
    - persistent_workers: 让worker不要每个epoch 都重启
    - pin_memory: 要不要把batch  放进页锁内存
    - IterableDataset:当数据不能随机访问


"""


class SimpleTensorDataset(utils.Dataset):

    def __init__(self, X: Tensor, Y: Tensor):
        if X.size(0) != Y.size(0):
            raise ValueError('X and Y must have the same size')
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        x = self.X[idx]
        y = self.Y[idx]
        return x, y


X = torch.randn(1000, 10)
y = torch.randn(1000, 1)

dataset = SimpleTensorDataset(X, y)
x0, y0 = dataset[0]

print('Dataset length:', len(dataset))
print('first element:', x0.shape)
print('second element:', y0.shape)

# DataLoader: 把样本组织成mini-batch

dataloader = utils.DataLoader(dataset, batch_size=2, shuffle=True)

X, y = next(iter(dataloader))
print('Input  batch shape:', X.shape)
print('Target  batch shape:', X.shape)


# for X,y in dataloader:
#     y_pred = model(X)
#     loss = mse_loss(y_pred, y)
#     loss.backward()


# collate_fn: 样本如何拼成batch


class VariableDataset(utils.Dataset):
    def __init__(self):
        self.samples = [
            torch.tensor([1, 2, 3]),
            torch.tensor([4, 5]),
            torch.tensor([6, 7, 8, 9]),
            torch.tensor([10])
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tensor:
        return self.samples[idx]


dataset = VariableDataset()
dataloader = utils.DataLoader(dataset, batch_size=2)

try:
    batch = next(iter(dataloader))
except RuntimeError as err:
    print('RuntimeError:', err)


def pad_collate(batch: list[Tensor]) -> tuple[Tensor, Tensor]:
    lengths = torch.tensor([len(x) for x in batch])
    max_len = lengths.max().item()
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, x in enumerate(batch):
        padded[i, :len(x)] = x
    return padded, lengths


dataloader = utils.DataLoader(dataset, batch_size=2, collate_fn=pad_collate)

for tokens, lenghts in dataloader:
    print('tokend', tokens, sep='\n')
    print('lenghts', lenghts, sep='\n\n')


# num_workers: 让数据加载和模型计算并行

# dataloader = utils.DataLoader(dataset, batch_size=2,collate_fn=pad_collate,num_workers=2)




PyTorch version: 2.13.0+cpu
PyTorch device: cpu
Dataset length: 1000
first element: torch.Size([10])
second element: torch.Size([1])
Input  batch shape: torch.Size([2, 10])
Target  batch shape: torch.Size([2, 10])
RuntimeError: stack expects each tensor to be equal size, but got [3] at entry 0 and [2] at entry 1
tokend
tensor([[1, 2, 3],
        [4, 5, 0]])
lenghts

tensor([3, 2])
tokend
tensor([[ 6,  7,  8,  9],
        [10,  0,  0,  0]])
lenghts

tensor([4, 1])


In [ ]:
# 正式的数据加载器


device = torch.accelerator.current_accelerator(check_available=True)

if device is None:
    device = torch.device('cpu')

dataset = utils.TensorDataset(torch.randn(1000, 10), torch.randn(1000, 1))

dataloader = utils.DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=device.type == 'cuda',
                              persistent_workers=True)

for X, y in dataloader:
    X = X.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


